# 02 - Feature Engineering

À partir du dataset préparé dans le notebook 01, on va :
1. Calculer le résultat et les points de chaque match
2. Construire les statistiques historiques par équipe (buts, points, classement adverse)
3. Créer des features différentielles entre l'équipe domicile et l'équipe extérieure
4. Sauvegarder le dataset final prêt pour l'entraînement du modèle

In [1]:
import pandas as pd
import numpy as np
import os

## 1. Chargement des données

In [2]:
# On charge le dataset produit par le notebook 01
df = pd.read_csv("../output/dataset_prepared.csv")
df["date"] = pd.to_datetime(df["date"])

print(f"Dataset chargé : {len(df)} matchs")
print(f"Colonnes : {list(df.columns)}")
df.tail()

Dataset chargé : 3152 matchs
Colonnes : ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'rank_home', 'total_points_home', 'rank_away', 'total_points_away']


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,rank_home,total_points_home,rank_away,total_points_away
3147,2026-03-31,Czech Republic,Denmark,2.0,2.0,FIFA World Cup qualification,Prague,Czech Republic,False,43.0,1487.0,21.0,1617.0
3148,2026-03-31,Peru,Honduras,2.0,2.0,Friendly,Madrid,Spain,True,53.0,1460.0,65.0,1380.0
3149,2026-03-31,United States,Portugal,0.0,2.0,Friendly,Atlanta,United States,False,15.0,1682.0,6.0,1760.0
3150,2026-03-31,Australia,Curaçao,5.0,1.0,FIFA Series,Melbourne,Australia,False,27.0,1574.0,81.0,1303.0
3151,2026-03-31,Kazakhstan,Comoros,1.0,0.0,FIFA Series,Astana,Kazakhstan,False,114.0,1173.0,106.0,1193.0


## 2. Résultat et points par match

On encode le résultat de chaque match :
- `0` : victoire domicile
- `1` : victoire extérieur
- `2` : match nul

Les points de match suivent la règle standard : 3 pour une victoire, 1 pour un nul, 0 pour une défaite.

In [3]:
# Fonction qui renvoie (résultat, points domicile, points extérieur) selon les buts
def result_finder(home, away):
    if home > away:
        return pd.Series([0, 3, 0])   # victoire domicile
    if home < away:
        return pd.Series([1, 0, 3])   # victoire extérieur
    else:
        return pd.Series([2, 1, 1])   # match nul

df[["result", "home_team_points", "away_team_points"]] = df.apply(
    lambda x: result_finder(x["home_score"], x["away_score"]), axis=1
)

print(df["result"].value_counts().rename({0: "Victoire domicile", 1: "Victoire extérieur", 2: "Nul"}))

result
Victoire domicile     1467
Victoire extérieur     939
Nul                    746
Name: count, dtype: int64


In [4]:
# Colonnes auxiliaires utilisées pour calculer les features
# rank_dif : différence de classement FIFA (positif = domicile moins bien classé)
df["rank_dif"] = df["rank_home"] - df["rank_away"]

# points par rang adverse : les points gagnés rapportés au classement de l'adversaire
# plus l'adversaire est bien classé (rang faible), plus le ratio est élevé
df["points_home_by_rank"] = df["home_team_points"] / df["rank_away"]
df["points_away_by_rank"] = df["away_team_points"] / df["rank_home"]

## 3. Statistiques historiques par équipe

Features calculées (sur tout le cycle et sur les 5 derniers matchs) :
- Moyenne de buts marqués / encaissés
- Classement moyen des adversaires affrontés
- Points FIFA gagnés
- Points de match moyens
- Points de match moyens par rang adverse

In [5]:
# On sépare le dataset en vue domicile et vue extérieure
# puis on les renomme de façon identique pour pouvoir les empiler

home_team = df[[
    "date", "home_team", "home_score", "away_score",
    "rank_home", "rank_away", "total_points_home",
    "result", "rank_dif", "points_home_by_rank", "home_team_points"
]]

away_team = df[[
    "date", "away_team", "away_score", "home_score",
    "rank_away", "rank_home", "total_points_away",
    "result", "rank_dif", "points_away_by_rank", "away_team_points"
]]

# On renomme les colonnes pour avoir la même structure dans les deux vues :
# "home_" → supprimé, "away_" → préfixe "suf_" (pour "suffered", i.e. l'adversaire)
home_team.columns = [
    h.replace("home_", "").replace("_home", "")
     .replace("away_", "suf_").replace("_away", "_suf")
    for h in home_team.columns
]
away_team.columns = [
    a.replace("away_", "").replace("_away", "")
     .replace("home_", "suf_").replace("_home", "_suf")
    for a in away_team.columns
]

# On empile les deux vues en une seule table par équipe
team_stats = pd.concat([home_team, away_team]).reset_index(drop=True)

# On sauvegarde cette table brute (sans stats calculées) pour la simulation 
team_stats_raw = team_stats.copy()

print(f"Table équipes : {len(team_stats)} lignes ({len(df)} matchs × 2 équipes)")
print(f"Colonnes : {list(team_stats.columns)}")

Table équipes : 6304 lignes (3152 matchs × 2 équipes)
Colonnes : ['date', 'team', 'score', 'suf_score', 'rank', 'rank_suf', 'total_points', 'result', 'rank_dif', 'points_by_rank', 'team_points']


In [6]:
# Pour chaque ligne (= une équipe dans un match), on calcule ses stats
# sur tous ses matchs précédents dans le cycle CDM 2026

stats_val = []

for index, row in team_stats.iterrows():
    team = row["team"]
    date = row["date"]

    # Tous les matchs passés de cette équipe, du plus récent au plus ancien
    past_games = team_stats[
        (team_stats["team"] == team) & (team_stats["date"] < date)
    ].sort_values("date", ascending=False)
    last5 = past_games.head(5)

    # Buts marqués et encaissés (moyenne sur tout le cycle et sur les 5 derniers)
    goals = past_games["score"].mean()
    goals_l5 = last5["score"].mean()
    goals_suf = past_games["suf_score"].mean()
    goals_suf_l5  = last5["suf_score"].mean()

    # Classement moyen des adversaires affrontés
    rank_suf = past_games["rank_suf"].mean()
    rank_suf_l5 = last5["rank_suf"].mean()

    # Points FIFA gagnés sur la période (différence entre le premier et le dernier)
    if len(past_games) > 0:
        points = past_games["total_points"].values[0] - past_games["total_points"].values[-1]
        points_l5 = last5["total_points"].values[0] - last5["total_points"].values[-1] if len(last5) > 0 else 0
    else:
        points, points_l5 = 0, 0

    # Points de match moyens (0/1/3) et points de match par rang adverse
    gp = past_games["team_points"].mean()
    gp_l5 = last5["team_points"].mean()
    gp_rank = past_games["points_by_rank"].mean()
    gp_rank_l5 = last5["points_by_rank"].mean()

    stats_val.append([
        goals, goals_l5, goals_suf, goals_suf_l5,
        rank_suf, rank_suf_l5,
        points, points_l5,
        gp, gp_l5,
        gp_rank, gp_rank_l5
    ])

print("Calcul terminé.")

Calcul terminé.


In [7]:
# On assemble les stats calculées avec la table équipes
stats_cols = [
    "goals_mean", "goals_mean_l5", "goals_suf_mean", "goals_suf_mean_l5",
    "rank_mean", "rank_mean_l5",
    "points_mean", "points_mean_l5",
    "game_points_mean", "game_points_mean_l5",
    "game_points_rank_mean", "game_points_rank_mean_l5"
]

stats_df  = pd.DataFrame(stats_val, columns=stats_cols)
full_df   = pd.concat([team_stats.reset_index(drop=True), stats_df], axis=1)

print(f"Table complète : {len(full_df)} lignes, {len(full_df.columns)} colonnes")

Table complète : 6304 lignes, 23 colonnes


In [8]:
# On re-sépare en vue domicile et vue extérieure pour récupérer les stats
# Les deux moitiés correspondent exactement à home_team et away_team empilés
n = len(df)
home_team_stats = full_df.iloc[:n, -12:].copy()
away_team_stats = full_df.iloc[n:, -12:].copy()

# On préfixe les colonnes avec home_ / away_ pour les distinguer dans le dataset final
home_team_stats.columns = ["home_" + c for c in home_team_stats.columns]
away_team_stats.columns = ["away_" + c for c in away_team_stats.columns]

# On fusionne avec le dataset original (réindexé pour l'alignement)
match_stats = pd.concat(
    [home_team_stats.reset_index(drop=True), away_team_stats.reset_index(drop=True)],
    axis=1
)
full_df = pd.concat([df.reset_index(drop=True), match_stats], axis=1)

print(f"Dataset avec stats : {len(full_df)} lignes, {len(full_df.columns)} colonnes")

Dataset avec stats : 3152 lignes, 43 colonnes


## 4. Features différentielles

Comme dans le notebook de référence, les features les plus discriminantes sont les **différences** entre les stats des deux équipes. On crée également une variable pour indiquer si le match est un match amical.

In [9]:
# Différences de buts marqués et encaissés
full_df["goals_dif"] = full_df["home_goals_mean"] - full_df["away_goals_mean"]
full_df["goals_dif_l5"] = full_df["home_goals_mean_l5"] - full_df["away_goals_mean_l5"]
full_df["goals_suf_dif"] = full_df["home_goals_suf_mean"] - full_df["away_goals_suf_mean"]
full_df["goals_suf_dif_l5"] = full_df["home_goals_suf_mean_l5"] - full_df["away_goals_suf_mean_l5"]

# Buts marqués par classement adverse : mesure la performance contre des équipes fortes
full_df["goals_per_ranking_dif"] = (
    full_df["home_goals_mean"] / full_df["home_rank_mean"]
  - full_df["away_goals_mean"] / full_df["away_rank_mean"]
)

# Différence de classement moyen des adversaires affrontés (indique le niveau du calendrier)
full_df["dif_rank_agst"] = full_df["home_rank_mean"] - full_df["away_rank_mean"]
full_df["dif_rank_agst_l5"] = full_df["home_rank_mean_l5"] - full_df["away_rank_mean_l5"]

# Différence de points de match par rang adverse (qualité des victoires)
full_df["dif_points_rank"] = full_df["home_game_points_rank_mean"] - full_df["away_game_points_rank_mean"]
full_df["dif_points_rank_l5"] = full_df["home_game_points_rank_mean_l5"] - full_df["away_game_points_rank_mean_l5"]

# Variable indiquant si le match est un amical (les matchs officiels ont plus d'enjeu)
full_df["is_friendly"] = (full_df["tournament"] == "Friendly").astype(int)

print("Features différentielles créées.")

Features différentielles créées.


## 5. Construction du dataset final

In [10]:
# Fonction reprise du notebook de référence :
# construit le dataset de modélisation à partir du dataset enrichi
def create_db(df):
    # Colonnes nécessaires pour calculer les features différentielles
    columns = [
        "home_team", "away_team", "result",
        "rank_dif",
        "home_goals_mean", "home_rank_mean", "away_goals_mean", "away_rank_mean",
        "home_rank_mean_l5", "away_rank_mean_l5",
        "home_goals_suf_mean", "away_goals_suf_mean",
        "home_goals_mean_l5", "away_goals_mean_l5",
        "home_goals_suf_mean_l5", "away_goals_suf_mean_l5",
        "home_game_points_rank_mean", "home_game_points_rank_mean_l5",
        "away_game_points_rank_mean", "away_game_points_rank_mean_l5",
        "is_friendly"
    ]

    base = df[columns].copy()

    # Encodage de la cible : nul (2) → 1 (domicile ne gagne pas)
    base["target"] = base["result"].apply(lambda x: 1 if x == 2 else x)

    # Features différentielles
    base["goals_dif"]              = base["home_goals_mean"]              - base["away_goals_mean"]
    base["goals_dif_l5"]           = base["home_goals_mean_l5"]           - base["away_goals_mean_l5"]
    base["goals_suf_dif"]          = base["home_goals_suf_mean"]          - base["away_goals_suf_mean"]
    base["goals_suf_dif_l5"]       = base["home_goals_suf_mean_l5"]       - base["away_goals_suf_mean_l5"]
    base["goals_per_ranking_dif"]  = (
        base["home_goals_mean"] / base["home_rank_mean"]
      - base["away_goals_mean"] / base["away_rank_mean"]
    )
    base["dif_rank_agst"]          = base["home_rank_mean"]    - base["away_rank_mean"]
    base["dif_rank_agst_l5"]       = base["home_rank_mean_l5"] - base["away_rank_mean_l5"]
    base["dif_points_rank"]        = base["home_game_points_rank_mean"]    - base["away_game_points_rank_mean"]
    base["dif_points_rank_l5"]     = base["home_game_points_rank_mean_l5"] - base["away_game_points_rank_mean_l5"]

    # On ne garde que les colonnes finales du modèle
    model_df = base[[
        "home_team", "away_team", "target",
        "rank_dif",
        "goals_dif", "goals_dif_l5",
        "goals_suf_dif", "goals_suf_dif_l5",
        "goals_per_ranking_dif",
        "dif_rank_agst", "dif_rank_agst_l5",
        "dif_points_rank", "dif_points_rank_l5",
        "is_friendly"
    ]]
    return model_df

In [11]:
model_db = create_db(full_df)

# On supprime les matchs pour lesquels les stats n'ont pas pu être calculées
# (premiers matchs du cycle sans historique)
model_db = model_db.dropna().reset_index(drop=True)

print(f"Dataset final : {len(model_db)} matchs, {len(model_db.columns)} colonnes")
print(f"Matchs écartés (pas d'historique) : {len(full_df) - len(model_db)}")
print(f"\nValeurs manquantes :\n{model_db.isna().sum()}")
model_db.tail()

Dataset final : 3019 matchs, 14 colonnes
Matchs écartés (pas d'historique) : 133

Valeurs manquantes :
home_team                0
away_team                0
target                   0
rank_dif                 0
goals_dif                0
goals_dif_l5             0
goals_suf_dif            0
goals_suf_dif_l5         0
goals_per_ranking_dif    0
dif_rank_agst            0
dif_rank_agst_l5         0
dif_points_rank          0
dif_points_rank_l5       0
is_friendly              0
dtype: int64


,home_team,away_team,target,rank_dif,goals_dif,goals_dif_l5,goals_suf_dif,goals_suf_dif_l5,goals_per_ranking_dif,dif_rank_agst,dif_rank_agst_l5,dif_points_rank,dif_points_rank_l5,is_friendly
3014,Czech Republic,Denmark,1,22.0,0.025210,-1.4,0.087395,-0.6,-0.008730,24.402521,53.4,-0.026026,-0.000563,0
3015,Peru,Honduras,1,-12.0,-0.829545,-0.4,0.001515,1.2,-0.002019,-46.964394,-48.8,0.004912,-0.013545,1
3016,United States,Portugal,1,9.0,-0.614202,0.0,0.299417,0.8,-0.008428,-3.961844,-34.4,-0.039586,0.054735,1
3017,Australia,Curaçao,0,-54.0,0.121212,-2.0,-0.420875,0.6,0.009220,-35.047138,-83.8,0.019255,0.015538,0
3018,Kazakhstan,Comoros,0,8.0,-0.137019,1.4,0.385417,-0.2,0.001371,-19.282051,33.6,0.005436,0.028534,0


In [12]:
# Distribution de la cible
print("Distribution de la cible :")
print(model_db["target"].value_counts().rename({0: "Victoire domicile", 1: "Pas de victoire domicile"}))

Distribution de la cible :
target
Pas de victoire domicile    1625
Victoire domicile           1394
Name: count, dtype: int64


## 6. Sauvegarde

In [13]:
os.makedirs("../output", exist_ok=True)

# Dataset de modélisation
model_db.to_csv("../output/dataset_features.csv", index=False)
print(f"Sauvegardé : ../output/dataset_features.csv ({len(model_db)} lignes)")

# Stats brutes par équipe (utilisées dans le notebook 04 pour la simulation)
team_stats_raw.to_csv("../output/team_stats_raw.csv", index=False)
print(f"Sauvegardé : ../output/team_stats_raw.csv ({len(team_stats_raw)} lignes)")

Sauvegardé : ../output/dataset_features.csv (3019 lignes)
Sauvegardé : ../output/team_stats_raw.csv (6304 lignes)
